# GraphCastSmall — Myanmar 48h Weather Forecast (Local M4 Pipeline)

**Model**: `earth2studio.models.px.GraphCastSmall` (DeepMind/Google, via NVIDIA Earth2Studio)  
**Resolution**: 1.0° global (181 × 360)  
**Timestep**: 6h native — no interpolation between steps  
**Horizon**: 48h (8 AR steps → t+6h, t+12h, ..., t+48h)  
**Variables**: `tp06` (6h accumulated precipitation) + `t2m` (2m temperature)  
**Transform**: tp06: physical metres × 1000 → mm/6h — **no log/exp transform**  
**Transform**: t2m: Kelvin − 273.15 → °C  
**Init source**: ARCO ERA5 (historical, 1959-2023) or IFS HRES (near-real-time)  
**Hardware**: Apple M4 CPU, JAX / XLA ARM64 — **no GPU required**

---

## Before you start

1. Install dependencies: `uv sync` in the repository root
2. Activate the uv environment: `source .venv/bin/activate` (or use `uv run`)
3. Run cells **in order**. The smoke test (Section 5) must pass before the full forecast.
4. For a near-real-time forecast, set `SOURCE = "ifs"` in Section 3.

## Dependency constraints

```
earth2studio[aurora,data] >= 0.17.0
xarray >= 2024.1.0, < 2026        # CRITICAL: xr.Dataset(ds) removed in xarray 2026+
graphcast @ git+https://github.com/google-deepmind/graphcast@08cf736
JAX_PLATFORM_NAME=cpu             # Must be set before any JAX import
```

## Output (schema v3.0)

```
data/forecast/
  temperature.bin    [9 × 21 × 11] float32  (t+0h..t+48h, 6h steps, °C)  8,316 bytes
  precipitation.bin  [9 × 21 × 11] float32  (t+0h=0.0, mm/6h)            8,316 bytes
  forecast.json      schema v3.0, is_demo=false
```

## Myanmar grid at 1.0°

- lat: 9.0°N to 29.0°N → 21 points (ascending south-to-north)
- lon: 92.0°E to 102.0°E → 11 points

## Section 0 — Environment verification

In [ ]:
import sys, os, platform, subprocess

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"  Python        : {sys.version.split()[0]}")
print(f"  Platform      : {platform.platform()}")
print(f"  Machine       : {platform.machine()}")
print(f"  Processor     : {platform.processor()}")

# Verify we are NOT trying to use MPS or CUDA — JAX CPU is the target
import os
# MUST be set before any JAX import
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

print(f"  JAX backend   : {os.environ['JAX_PLATFORM_NAME']} (XLA ARM64)")
print()

# Check key package versions
import importlib
for pkg in ["earth2studio", "xarray", "numpy", "zarr", "jax", "psutil"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:20s}: {m.__version__}")
    except ImportError:
        print(f"  {pkg:20s}: NOT FOUND")

# Critical: xarray must be < 2026
import xarray as xr
xr_year = int(xr.__version__.split('.')[0])
if xr_year >= 2026:
    raise RuntimeError(
        f"xarray {xr.__version__} >= 2026 is INCOMPATIBLE with earth2studio 0.17.0. "
        "Pin: xarray>=2024.1.0,<2026 in pyproject.toml and run uv sync."
    )
print(f"\n  xarray version {xr.__version__} < 2026: OK")
print("=" * 60)

## Section 1 — Pipeline constants

These must match `scripts/generate_forecast.py` exactly.

In [ ]:
import numpy as np
import psutil

# ── Myanmar spatial constants ────────────────────────────────────────────────
MYANMAR_LAT_MIN = 9.0
MYANMAR_LAT_MAX = 29.0
MYANMAR_LON_MIN = 92.0
MYANMAR_LON_MAX = 102.0

# ── GraphCastSmall temporal constants ────────────────────────────────────────
GC_STEP_HOURS    = 6
GC_HORIZON_HOURS = 48
GC_N_STEPS       = GC_HORIZON_HOURS // GC_STEP_HOURS   # 8 AR steps
GC_N_FRAMES      = GC_N_STEPS + 1                      # 9 total (t+0h through t+48h)
GC_N_LAT         = 21   # 9°N to 29°N at 1.0°
GC_N_LON         = 11   # 92°E to 102°E at 1.0°

SANITY_MAX_MM    = 500.0   # mm/6h physical upper bound
TEMP_MIN_C       = -90.0
TEMP_MAX_C       = 70.0

def mem_gb():
    return psutil.Process().memory_info().rss / 1e9

print(f"Horizon   : {GC_HORIZON_HOURS}h ({GC_N_STEPS} AR steps × {GC_STEP_HOURS}h)")
print(f"Frames    : {GC_N_FRAMES} (t+0h through t+{GC_HORIZON_HOURS}h)")
print(f"Grid      : {GC_N_LAT} lat × {GC_N_LON} lon at 1.0°")
print(f"Variables : tp06 (metres → mm/6h) + t2m (K → °C)")
print(f"RSS start : {mem_gb():.2f} GB")

## Section 2 — Initialization source

- **ARCO** (default): ERA5 reanalysis on Google Cloud, 1959–2023, free, no credentials.
  Use any historical date within this range.
- **IFS**: ECMWF open data, near-real-time, free, no credentials.
  Use for a live operational forecast (last ~48h).

GraphCastSmall requires **two consecutive time steps** (t−6h and t+0h) as initialization.
Earth2Studio's `e2run.deterministic` handles this automatically.

In [ ]:
from datetime import datetime, timedelta, timezone

# ── Choose source and init time ──────────────────────────────────────────────
SOURCE        = "arco"                          # "arco" or "ifs"
INIT_TIME_STR = "2022-07-01T00:00:00Z"          # Myanmar monsoon; within ARCO 1959-2023
# For IFS live run: SOURCE = "ifs"; INIT_TIME_STR = None

if SOURCE == "arco":
    from earth2studio.data import ARCO
    data_source = ARCO()
    INIT_TIME = datetime.fromisoformat(INIT_TIME_STR.replace("Z", "+00:00"))
    SOURCE_LABEL = "ARCO ERA5 reanalysis (Google Cloud, 1959-2023, no credentials)"
    SOURCE_ATTR  = (
        "ERA5 via ARCO (Analysis-Ready Cloud-Optimized ERA5). "
        "Hosted on Google Cloud. ERA5 © ECMWF/Copernicus."
    )

elif SOURCE == "ifs":
    from earth2studio.data import IFS
    data_source = IFS()
    if INIT_TIME_STR:
        INIT_TIME = datetime.fromisoformat(INIT_TIME_STR.replace("Z", "+00:00"))
    else:
        # Auto-detect latest available IFS run (00Z or 12Z, ~6h lag)
        now = datetime.now(timezone.utc)
        for d in range(3):
            for h in [12, 0]:
                t = (now - timedelta(days=d)).replace(
                    hour=h, minute=0, second=0, microsecond=0
                )
                if (now - t).total_seconds() >= 6 * 3600:
                    INIT_TIME = t
                    break
            else:
                continue
            break
    SOURCE_LABEL = "IFS HRES open data (ECMWF, near-real-time, no credentials)"
    SOURCE_ATTR  = (
        "IFS HRES analysis — ECMWF open data. "
        "CC BY 4.0. https://confluence.ecmwf.int/display/DAC/ECMWF+open+data"
    )
else:
    raise ValueError(f"Unknown SOURCE: {SOURCE!r}. Use 'arco' or 'ifs'.")

print(f"Source    : {SOURCE_LABEL}")
print(f"Init time : {INIT_TIME.isoformat()}")
print(f"Note      : Earth2Studio will automatically fetch t-6h and t+0h.")

## Section 3 — Load GraphCastSmall

Downloads checkpoint from `gs://dm_graphcast/graphcast` (~300 MB on first run; cached thereafter).  
**Do NOT call `model.to(bfloat16)`** — GraphCastSmall applies bfloat16 internally via JAX's
`casting.Bfloat16Cast`. The model uses bfloat16 during inference regardless.

In [ ]:
import time
from earth2studio.models.px import GraphCastSmall

mem_before_load = mem_gb()
t_load = time.time()

print("Loading GraphCastSmall checkpoint...")
print("  Checkpoint: GraphCast_small - ERA5 1979-2015 - resolution 1.0 -")
print("              pressure levels 13 - mesh 2to5 - precipitation input and output.npz")
print("  Backend: JAX + Haiku (bfloat16 internal via casting.Bfloat16Cast)")
print()

package = GraphCastSmall.load_default_package()
model   = GraphCastSmall.load_model(package)

load_time = time.time() - t_load
mem_after_load = mem_gb()

print(f"Model loaded in {load_time:.1f}s")
print(f"RSS: {mem_after_load:.2f} GB (+{mem_after_load - mem_before_load:.2f} GB from model weights)")

## Section 4 — Smoke test (nsteps=1, ~10s)

Verifies the full pipeline compiles and runs without error before committing to the full 8-step run.
This includes JAX JIT compilation (first call only) and a single ARCO data fetch.

If this cell fails, diagnose the error before proceeding.

In [ ]:
import torch
import earth2studio.run as e2run
from earth2studio.io import ZarrBackend

SMOKE_PASSED = False

print("[SMOKE TEST] nsteps=1 — single 6h inference step")
print(f"  Backend: JAX CPU (XLA ARM64) | RSS before: {mem_gb():.2f} GB")

io_smoke = ZarrBackend()
t_smoke  = time.time()

try:
    with torch.inference_mode():
        io_smoke = e2run.deterministic(
            time=[INIT_TIME],
            nsteps=1,
            prognostic=model,
            data=data_source,
            io=io_smoke,
            device=torch.device("cpu"),
            verbose=True,
        )
    smoke_time = time.time() - t_smoke
    SMOKE_PASSED = True
    print(f"\nSMOKE TEST PASSED in {smoke_time:.1f}s")
    print(f"RSS after smoke: {mem_gb():.2f} GB")
    print("Proceeding to full 48h forecast.")

except Exception as e:
    elapsed = time.time() - t_smoke
    print(f"\nSMOKE TEST FAILED after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    print()
    print("Common causes:")
    print("  - JAX not using CPU: check JAX_PLATFORM_NAME=cpu was set before import")
    print("  - xarray >= 2026: run uv sync to enforce xarray<2026 pin")
    print("  - graphcast submodules missing: check pyproject.toml [tool.uv.sources]")
    print("  - ARCO/IFS connectivity: check network access to Google Cloud")
    raise

## Section 5 — Full 48h forecast (nsteps=8, ~54s)

Runs the full 8-step autoregressive rollout. After JIT compilation in Section 4,
subsequent steps run without recompilation (~6s each on M4 CPU).

In [ ]:
assert SMOKE_PASSED, "Smoke test must pass before full run"

FULL_PASSED = False

print(f"[FULL FORECAST] nsteps={GC_N_STEPS} — {GC_N_STEPS} × {GC_STEP_HOURS}h = {GC_HORIZON_HOURS}h")
print(f"  RSS before inference: {mem_gb():.2f} GB")

io_full  = ZarrBackend()
t_infer  = time.time()
mem_pre  = mem_gb()

try:
    with torch.inference_mode():
        io_full = e2run.deterministic(
            time=[INIT_TIME],
            nsteps=GC_N_STEPS,
            prognostic=model,
            data=data_source,
            io=io_full,
            device=torch.device("cpu"),
            verbose=True,
        )
    inference_time = time.time() - t_infer
    mem_post = mem_gb()
    FULL_PASSED = True
    print(f"\nFULL FORECAST PASSED in {inference_time:.1f}s ({inference_time/60:.1f} min)")
    print(f"RSS: {mem_post:.2f} GB (+{mem_post - mem_pre:.2f} GB during inference)")

except Exception as e:
    elapsed = time.time() - t_infer
    print(f"\nFULL FORECAST FAILED after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    raise

# Free model from memory
del model
print(f"Model released | RSS: {mem_gb():.2f} GB")

## Section 6 — Post-process: tp06 → mm/6h

**Pipeline (NO log/exp transform):**
```
GraphCastSmall zarr output: tp06 in physical metres
    zarr shape: (n_init=1, n_frames=9, 181, 360)
    ↓  root["tp06"][0]  →  (9, 181, 360)
    ↓  lat ascending sort + Myanmar subset  →  (9, 21, 11)
    ↓  × 1000  →  mm / 6h accumulation
    ↓  clamp ≥ 0  (physical constraint)
    ↓  t+0h frame[0] = 0.0  (no forecast accumulation at init hour)
output: float32 [9, 21, 11]
```

In [ ]:
assert FULL_PASSED

root   = io_full.root
coords = io_full.coords

print(f"Zarr variables : {list(root.keys())}")
print(f"Coords         : {list(coords.keys())}")

for var in ("tp06", "t2m"):
    if var not in root:
        raise ValueError(f"'{var}' not found in zarr. Available: {list(root.keys())}")

# Raw arrays — shape (n_init=1, n_frames, 181, 360)
tp06_raw = root["tp06"][:]
t2m_raw  = root["t2m"][:]
print(f"tp06 raw shape : {tp06_raw.shape}  dtype: {tp06_raw.dtype}")
print(f"t2m  raw shape : {t2m_raw.shape}  dtype: {t2m_raw.dtype}")

# Drop init-time dimension
tp06_global = tp06_raw[0]   # (9, 181, 360)
t2m_global  = t2m_raw[0]

# Lat/lon coordinate arrays
lat_arr = np.array(coords["lat"], dtype=np.float64)   # (181,)
lon_arr = np.array(coords["lon"], dtype=np.float64)   # (360,)
print(f"Lat range      : {lat_arr.min():.1f} to {lat_arr.max():.1f}")
print(f"Lon range      : {lon_arr.min():.1f} to {lon_arr.max():.1f}")

# Myanmar lat/lon indices
lat_mask = (lat_arr >= MYANMAR_LAT_MIN) & (lat_arr <= MYANMAR_LAT_MAX)
lon_mask = (lon_arr >= MYANMAR_LON_MIN) & (lon_arr <= MYANMAR_LON_MAX)
lat_idx  = np.where(lat_mask)[0]
lon_idx  = np.where(lon_mask)[0]

# Sort lat ascending (south→north) regardless of zarr storage order
sort_order  = np.argsort(lat_arr[lat_idx])
lat_idx_asc = lat_idx[sort_order]
lats_out    = lat_arr[lat_idx_asc]
lons_out    = lon_arr[lon_idx]

print(f"Myanmar grid   : {len(lats_out)} lat × {len(lons_out)} lon (expected 21 × 11)")
print(f"Lat            : {lats_out[0]:.1f}°N to {lats_out[-1]:.1f}°N")
print(f"Lon            : {lons_out[0]:.1f}°E to {lons_out[-1]:.1f}°E")

# Subset to Myanmar
tp06_myanmar = tp06_global[:, lat_idx_asc, :][:, :, lon_idx]   # (9, 21, 11)
t2m_myanmar  = t2m_global[:,  lat_idx_asc, :][:, :, lon_idx]   # (9, 21, 11)

# tp06: physical metres → mm/6h (NO exp/log transform)
tp06_mm = np.maximum(tp06_myanmar * 1000.0, 0.0).astype(np.float32)
tp06_mm[0] = 0.0   # t+0h: no forecast accumulation at init hour

print(f"\ntp06 metres raw: [{tp06_myanmar.min():.6f}, {tp06_myanmar.max():.4f}]")
print(f"tp06 mm/6h     : [{tp06_mm.min():.4f}, {tp06_mm.max():.4f}]")
print(f"tp06 shape     : {tp06_mm.shape}  dtype: {tp06_mm.dtype}")

## Section 7 — Post-process: t2m → °C

**Pipeline:**
```
GraphCastSmall zarr output: t2m in Kelvin
    ↓  K − 273.15  →  °C
output: float32 [9, 21, 11]
```

t+0h frame is the analysis temperature from the initialization source (not a forecast).

In [ ]:
# t2m: Kelvin → Celsius
t2m_c = (t2m_myanmar - 273.15).astype(np.float32)

print(f"t2m Kelvin raw : [{t2m_myanmar.min():.2f}, {t2m_myanmar.max():.2f}]")
print(f"t2m °C         : [{t2m_c.min():.2f}, {t2m_c.max():.2f}]")
print(f"t2m shape      : {t2m_c.shape}  dtype: {t2m_c.dtype}")

# Per-frame summary
print("\nPer-frame temperature range (°C):")
for i in range(GC_N_FRAMES):
    note = " ← analysis (not forecast)" if i == 0 else ""
    print(f"  t+{i*GC_STEP_HOURS:2d}h [{t2m_c[i].min():.2f}, {t2m_c[i].max():.2f}]{note}")

## Section 8 — Quality control

In [ ]:
PRECIP_QC_PASSED = False
TEMP_QC_PASSED   = False

# ── Precipitation QC ─────────────────────────────────────────────────────────
print("PRECIPITATION QC (tp06)")
print("-" * 40)
p_nan = int(np.sum(np.isnan(tp06_mm)))
p_inf = int(np.sum(np.isinf(tp06_mm)))
p_neg = int(np.sum(tp06_mm < 0.0))
p_valid = tp06_mm[np.isfinite(tp06_mm)]
print(f"  Shape   : {tp06_mm.shape}")
print(f"  Min     : {p_valid.min():.4f} mm/6h")
print(f"  Median  : {float(np.median(p_valid)):.4f} mm/6h")
print(f"  P95     : {float(np.percentile(p_valid, 95)):.4f} mm/6h")
print(f"  Max     : {p_valid.max():.4f} mm/6h")
print(f"  NaN     : {p_nan}")
print(f"  Inf     : {p_inf}")
print(f"  Neg     : {p_neg}")
p_failures = []
if p_nan > 0: p_failures.append(f"NaN: {p_nan}")
if p_inf > 0: p_failures.append(f"Inf: {p_inf}")
if p_neg > 0: p_failures.append(f"Negative: {p_neg}")
if p_valid.max() > SANITY_MAX_MM: p_failures.append(f"Max > {SANITY_MAX_MM} mm/6h")
PRECIP_QC_PASSED = not p_failures
print(f"  Status  : {'PASS' if PRECIP_QC_PASSED else 'FAIL — ' + str(p_failures)}")

print()
# ── Temperature QC ───────────────────────────────────────────────────────────
print("TEMPERATURE QC (t2m)")
print("-" * 40)
t_nan = int(np.sum(np.isnan(t2m_c)))
t_inf = int(np.sum(np.isinf(t2m_c)))
t_valid = t2m_c[np.isfinite(t2m_c)]
print(f"  Shape   : {t2m_c.shape}")
print(f"  Min     : {t_valid.min():.2f} °C")
print(f"  Mean    : {float(np.mean(t_valid)):.2f} °C")
print(f"  Max     : {t_valid.max():.2f} °C")
print(f"  NaN     : {t_nan}")
print(f"  Inf     : {t_inf}")
t_failures = []
if t_nan > 0: t_failures.append(f"NaN: {t_nan}")
if t_inf > 0: t_failures.append(f"Inf: {t_inf}")
if t_valid.min() < TEMP_MIN_C: t_failures.append(f"Min < {TEMP_MIN_C}°C")
if t_valid.max() > TEMP_MAX_C: t_failures.append(f"Max > {TEMP_MAX_C}°C")
TEMP_QC_PASSED = not t_failures
print(f"  Status  : {'PASS' if TEMP_QC_PASSED else 'FAIL — ' + str(t_failures)}")

## Section 9 — Write artifacts (schema v3.0)

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
import earth2studio

OUTPUT_DIR = Path("data/forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

times_utc = [
    (INIT_TIME + timedelta(hours=i * GC_STEP_HOURS)).strftime("%Y-%m-%dT%H:%M:%SZ")
    for i in range(GC_N_FRAMES)
]

# ── temperature.bin ──────────────────────────────────────────────────────────
temp_path = OUTPUT_DIR / "temperature.bin"
temp_path.write_bytes(t2m_c.astype("<f4").tobytes())
assert temp_path.stat().st_size == GC_N_FRAMES * GC_N_LAT * GC_N_LON * 4
print(f"temperature.bin   : {temp_path.stat().st_size:,} bytes")

# ── precipitation.bin ────────────────────────────────────────────────────────
precip_path = OUTPUT_DIR / "precipitation.bin"
precip_path.write_bytes(tp06_mm.astype("<f4").tobytes())
assert precip_path.stat().st_size == GC_N_FRAMES * GC_N_LAT * GC_N_LON * 4
print(f"precipitation.bin : {precip_path.stat().st_size:,} bytes")

# ── forecast.json (schema v3.0) ──────────────────────────────────────────────
total_time = load_time + inference_time
peak_rss   = mem_gb()

meta = {
    "schema_version": "3.0",
    "model": "GraphCastSmall",
    "model_version": "1.0",
    "model_checkpoint": (
        "GraphCast_small - ERA5 1979-2015 - resolution 1.0 - "
        "pressure levels 13 - mesh 2to5 - precipitation input and output.npz"
    ),
    "model_source": "gs://dm_graphcast/graphcast",
    "model_attribution": (
        "GraphCast by DeepMind/Google. Lam et al. (2023), Science. "
        "https://arxiv.org/abs/2212.12794. Earth2Studio wrapper by NVIDIA."
    ),
    "initialization_source": SOURCE.upper(),
    "initialization_time": times_utc[0],
    "forecast_generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "forecast_horizon_hours": GC_HORIZON_HOURS,
    "native_timestep_hours": GC_STEP_HOURS,
    "n_times": GC_N_FRAMES,
    "spatial_resolution_deg": 1.0,
    "display_resolution_deg": None,
    "region": "Myanmar",
    "bbox": {
        "lat_min": float(lats_out.min()), "lat_max": float(lats_out.max()),
        "lon_min": float(lons_out.min()), "lon_max": float(lons_out.max()),
    },
    "grid": {"n_lat": len(lats_out), "n_lon": len(lons_out)},
    "lat": lats_out.tolist(),
    "lon": lons_out.tolist(),
    "times_utc": times_utc,
    "variables": {
        "precipitation": {
            "display_name": "Precipitation",
            "units": "mm / 6h",
            "source_variable": "tp06",
            "temporal_resolution": "6-hourly",
            "temporal_semantics": (
                "Total precipitation accumulated over the 6-hour forecast "
                "period ending at the displayed valid time."
            ),
            "temporal_disclosure": (
                "Precipitation values represent total rainfall accumulated "
                "during the 6-hour forecast period ending at the displayed time. "
                "These are not instantaneous rainfall rates."
            ),
            "transformation_provenance": {
                "source_variable": "tp06",
                "source_unit": "metres",
                "conversion": "metres × 1000",
                "output_unit": "mm",
                "accumulation_period_hours": GC_STEP_HOURS,
                "log_transform_applied": False,
                "exp_transform_applied": False,
                "pipeline": (
                    "GraphCastSmall native tp06 (physical metres, no log transform) "
                    "→ metres × 1000 → mm/6h → clamp ≥ 0"
                ),
            },
            "t0_note": (
                "t+0h frame (index 0) is set to 0.0 mm/6h. "
                "Represents the analysis state; no forecast accumulation at init hour."
            ),
            "native_output": True,
            "file": "precipitation.bin",
            "fill_value": None,
        },
        "temperature": {
            "display_name": "Temperature",
            "units": "\u00b0C",
            "source_variable": "t2m",
            "temporal_resolution": "6-hourly",
            "temporal_semantics": "2m temperature at the forecast valid time.",
            "transformation_provenance": {
                "source_variable": "t2m",
                "source_unit": "K",
                "conversion": "K - 273.15",
                "output_unit": "\u00b0C",
                "log_transform_applied": False,
                "exp_transform_applied": False,
                "pipeline": "GraphCastSmall native t2m (Kelvin) → K - 273.15 → °C",
            },
            "t0_note": (
                "t+0h frame (index 0) is the analysis temperature from the data source, "
                "not a forecast."
            ),
            "native_output": True,
            "file": "temperature.bin",
            "fill_value": None,
        },
    },
    "data_source_attribution": SOURCE_ATTR,
    "earth2studio_version": earth2studio.__version__,
    "inference_config": {
        "device": "Apple M4 CPU",
        "jax_backend": "cpu",
        "jax_env": {
            "XLA_PYTHON_CLIENT_PREALLOCATE": os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"),
            "JAX_PLATFORM_NAME": os.environ.get("JAX_PLATFORM_NAME"),
        },
        "rss_peak_gb": round(peak_rss, 2),
        "model_load_time_seconds": round(load_time),
        "inference_time_seconds": round(inference_time),
        "total_pipeline_time_seconds": round(total_time),
    },
    "is_demo": False,
}

json_path = OUTPUT_DIR / "forecast.json"
with open(json_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"forecast.json     : {json_path.stat().st_size:,} bytes")
print(f"\nArtifacts in {OUTPUT_DIR.resolve()}:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}: {p.stat().st_size:,} bytes")

## Section 10 — Validate artifacts (25 checks)

Must all PASS before proceeding.

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "scripts/validate_forecast.py", "--data-dir", str(OUTPUT_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"Validation FAILED (exit {result.returncode}) — do not push.")

VALIDATE_PASSED = True
print("All 25 validation checks passed. Artifacts are ready for deployment.")

## Section 11 — Provenance report

In [ ]:
print("=" * 62)
print("GRAPHCASTSMALL M4 CPU PROVENANCE REPORT")
print("=" * 62)
print()
print("MODEL")
print(f"  Name              : GraphCastSmall (Earth2Studio {earth2studio.__version__})")
print(f"  Native resolution : 1.0° (181 × 360 global)")
print(f"  Backend           : JAX CPU / XLA ARM64")
print()
print("INITIALIZATION")
print(f"  Source            : {SOURCE.upper()}")
print(f"  Init time         : {INIT_TIME.isoformat()}")
print(f"  Input timesteps   : t-6h + t+0h (two-timestep requirement, auto via e2run)")
print()
print("FORECAST")
print(f"  Horizon           : {GC_HORIZON_HOURS}h")
print(f"  Step              : {GC_STEP_HOURS}h")
print(f"  Frames            : {GC_N_FRAMES} (t+0h .. t+{GC_HORIZON_HOURS}h)")
print(f"  Times             : {', '.join(times_utc)}")
print()
print("HARDWARE")
print(f"  Device            : Apple M4 CPU (JAX XLA ARM64)")
print(f"  Peak RSS          : {peak_rss:.2f} GB")
print(f"  Model load time   : {load_time:.1f}s")
print(f"  Inference (8 step): {inference_time:.1f}s")
print(f"  Total pipeline    : {total_time:.1f}s")
print()
print("PRECIPITATION (tp06 → mm/6h)")
print(f"  Variable          : tp06")
print(f"  Unit              : mm / 6h accumulation")
print(f"  Transform         : metres × 1000 (NO exp/log)")
print(f"  Min               : {float(tp06_mm.min()):.4f} mm/6h")
print(f"  Max               : {float(tp06_mm.max()):.4f} mm/6h")
print(f"  NaN / Neg         : {int(np.sum(np.isnan(tp06_mm)))} / {int(np.sum(tp06_mm < 0))}")
print()
print("TEMPERATURE (t2m → °C)")
print(f"  Variable          : t2m")
print(f"  Unit              : °C")
print(f"  Transform         : K − 273.15")
print(f"  Min               : {float(t2m_c.min()):.2f} °C")
print(f"  Max               : {float(t2m_c.max()):.2f} °C")
print(f"  NaN               : {int(np.sum(np.isnan(t2m_c)))}")
print()
print("OUTPUT")
print(f"  precipitation.bin : {precip_path.stat().st_size:,} bytes  {tp06_mm.shape}")
print(f"  temperature.bin   : {temp_path.stat().st_size:,} bytes  {t2m_c.shape}")
print(f"  forecast.json     : {json_path.stat().st_size:,} bytes")
print()
print("STATUS")
qc_p = 'PASS' if PRECIP_QC_PASSED else 'FAIL'
qc_t = 'PASS' if TEMP_QC_PASSED   else 'FAIL'
val  = 'PASS' if VALIDATE_PASSED   else 'FAIL'
print(f"  Precipitation QC : {qc_p}")
print(f"  Temperature QC   : {qc_t}")
print(f"  Artifact validation (25 checks): {val}")
print("=" * 62)

## Section 12 — Optional: git commit and push

Commits `data/forecast/` artifacts to `main`, triggering GitHub Actions → GitHub Pages deployment.

Run from the repository root. Requires git to be configured with push access.

In [ ]:
import subprocess, json as _json

# Only run this cell if validation passed
assert VALIDATE_PASSED, "Validation must pass before pushing"

meta_r   = _json.loads(open("data/forecast/forecast.json").read())
init_t   = meta_r["initialization_time"]
device_r = meta_r["inference_config"]["device"]
inf_s    = meta_r["inference_config"]["inference_time_seconds"]
total_s  = meta_r["inference_config"]["total_pipeline_time_seconds"]

# Stage artifacts
subprocess.run(
    ["git", "add",
     "data/forecast/precipitation.bin",
     "data/forecast/temperature.bin",
     "data/forecast/forecast.json"],
    check=True
)

commit_msg = (
    f"feat: GraphCastSmall 48h Myanmar forecast {init_t}\n\n"
    f"Model: GraphCastSmall 1.0° / 6h steps / tp06+t2m / schema v3.0\n"
    f"Source: {SOURCE.upper()}\n"
    f"Hardware: {device_r}\n"
    f"Inference: {inf_s}s | Total: {total_s}s\n"
    f"Shape: [9, 21, 11] float32 / 8316 bytes each\n"
)

cr = subprocess.run(["git", "commit", "-m", commit_msg], capture_output=True, text=True)
print(cr.stdout.strip() or cr.stderr.strip())

push = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)
if push.returncode == 0:
    print("Pushed to main. GitHub Actions will deploy to GitHub Pages.")
    print("  Actions: https://github.com/JiayanLim/myanmar-weather-forecast/actions")
    print("  Pages  : https://jiayanlim.github.io/myanmar-weather-forecast/")
else:
    print("Push failed:", push.stderr)